# N and greenhouse gas budgets

Converts the gap-filled fluxes to budget units, cumulates them over the
defined periods, propagates the flux uncertainty, and derives the N budget,
the N&#8322;O emission factors and the net greenhouse gas exchange per treatment.

**Input** `../data/raw/fluxes_meteo_mgmt.parquet`
**Output** printed budget tables reported in the manuscript


# IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd # pandas allows to do a lot of basic data manipulation such as loading csv data files
from datetime import datetime
import matplotlib.pyplot as plt # matplotlib is a basic ploting library of Python
from diive.core.io.files import load_parquet
import statsmodels.api as sm

# READ DATA

In [ ]:
# Load data
filename = "../data/raw/fluxes_meteo_mgmt.parquet"
data = load_parquet(filename)

# Set time interval
data = data['2023-10-15':'2024-08-22'].copy()

data

# CALCULATE BUDGETS

In [ ]:
# --- BUDGET FUNCTION ---
import re

def _ensemble_cumulative_sigma(subset, col_base):
    """
    Correlation-aware structural (ensemble) uncertainty

    CV models'predicted trajectories are summed first (one cumulative total
    per fold), and the spread taken across those totals
    """
    ens_cols = sorted(c for c in subset.columns if re.fullmatch(rf"{re.escape(col_base)}_ens\d+", c))
    if not ens_cols:
        raise KeyError(
            f"No '{col_base}_ens##' columns found in this dataframe -- the upstream "
            f"merge needs to have used merge_gapfill_results(..., store_uncertainty_inputs=True) "
            f"for the cumulative ensemble uncertainty to be reconstructable here."
        )
    flag_col = f"FLAG_{col_base}_ISFILLED"
    if flag_col not in subset.columns:
        raise KeyError(f"{flag_col!r} not found; needed to know which rows were gap-filled.")
    is_gap = subset[flag_col].astype(bool).to_numpy()

    preds_ens = subset[ens_cols].to_numpy().T            # (K, T)
    masked = np.where(is_gap[None, :], preds_ens, 0.0)   # zero out observed rows
    cum_paths = np.nancumsum(masked, axis=1)             # (K, T) -- each fold's running total
    return float(np.std(cum_paths[:, -1], ddof=1))        # spread of the K period TOTALS


def calculate_seasonal_budget(df, start_date, end_date, target_base, model_type, view, qc, factor):
    
    # Subset
    subset = df.loc[start_date:end_date].copy()
    n_days = (datetime.strptime(end_date, '%Y-%m-%d') - datetime.strptime(start_date, '%Y-%m-%d')).days + 1
    # Define the columns for all 2 cuts
    col_16 = f'{target_base}_L3.3_CUT_16_{qc}_{view}_gf{model_type}'
    col_50 = f'{target_base}_L3.3_CUT_50_{qc}_{view}_gf{model_type}'
    col_84 = f'{target_base}_L3.3_CUT_84_{qc}_{view}_gf{model_type}'
    # Uncertainty inputs (CUT_50 used as the representative estimate, same as before)
    col_sigmaObs_50 = f'{col_50}_sigmaObs'
    col_sigmaResid_50 = f'{col_50}_sigmaResid'
    col_isfilled_50 = f'FLAG_{col_50}_ISFILLED'

    # Calculate sums for each ustar threshold
    sum_16 = subset[col_16].sum() * factor
    sum_50 = subset[col_50].sum() * factor
    sum_84 = subset[col_84].sum() * factor

    # --- Calculate Uncertainties ---
    is_gap = subset[col_isfilled_50].astype(bool)

    # sigma_obs (measurement random error) and sigma_resid (gap-filling
    # residual scale) are assumed independent row-to-row, so a plain
    # quadrature sum over the period is correct for these two terms.
    sigma_obs_sq_sum = np.nansum(subset.loc[~is_gap, col_sigmaObs_50] ** 2)
    sigma_resid_sq_sum = np.nansum(subset.loc[is_gap, col_sigmaResid_50] ** 2)

    # sigma_ens is NOT independent row-to-row
    sigma_ens_cum = _ensemble_cumulative_sigma(subset, col_50)

    # Each component on its own, same (converted) units as Budget
    error_obs   = np.sqrt(sigma_obs_sq_sum) * factor
    error_resid = np.sqrt(sigma_resid_sq_sum) * factor
    error_ens   = sigma_ens_cum * factor

    # Random + Gap-filling Uncertainty (correlation-aware for the ensemble term, quadrature for the other two)
    sigma_rand = np.sqrt(error_obs**2 + error_resid**2 + error_ens**2)
    
    # Ustar Threshold Uncertainty (Systematic)
    # Defined as half the range between the lowest and highest estimate: (Max - Min) / 2
    all_sums = [sum_16, sum_50, sum_84]
    sigma_ustar = (max(all_sums) - min(all_sums)) / 2
    
    # Total Combined Uncertainty
    # Combine independent errors in quadrature: sqrt(sigma_rand^2 + sigma_ustar^2)
    sigma_total = np.sqrt(sigma_rand**2 + sigma_ustar**2)

    # Variance-fraction breakdown: how much of Error_Total^2 each component
    # accounts for. These four sum to 1.0 (100%) exactly.
    total_var = sigma_total**2 if sigma_total > 0 else np.nan
    varfrac_obs   = error_obs**2   / total_var
    varfrac_resid = error_resid**2 / total_var
    varfrac_ens   = error_ens**2   / total_var
    varfrac_ustar = sigma_ustar**2 / total_var

    # Seasonal 95% CI, expressed as a half-width around the budget
    ci_halfwidth = 1.96 * sigma_total

    # Daily (mean) budget + uncertainties + half-width (scaled by days)
    daily_sum_50 = sum_50 / n_days
    daily_error_obs    = error_obs    / n_days
    daily_error_resid  = error_resid  / n_days
    daily_error_ens    = error_ens    / n_days
    daily_sigma_rand   = sigma_rand   / n_days
    daily_sigma_ustar  = sigma_ustar  / n_days
    daily_sigma_total  = sigma_total  / n_days
    daily_ci_halfwidth = ci_halfwidth / n_days

    return {
        'View': view,
        'QC': qc,

        'Budget': sum_50,

        # Individual uncertainty components (absolute, same units as Budget)
        'Error_Obs': error_obs,
        'Error_Resid': error_resid,
        'Error_Ens': error_ens,
        'Error_Random': sigma_rand,     # = sqrt(Obs^2 + Resid^2 + Ens^2)
        'Error_Ustar': sigma_ustar,
        'Error_Total': sigma_total,     # = sqrt(Error_Random^2 + Error_Ustar^2)

        # Share of TOTAL variance from each component (sums to 1.0)
        'VarFrac_Obs': varfrac_obs,
        'VarFrac_Resid': varfrac_resid,
        'VarFrac_Ens': varfrac_ens,
        'VarFrac_Ustar': varfrac_ustar,

        'CI_95_HalfWidth': ci_halfwidth,

        'Daily_Budget': daily_sum_50,
        'Daily_Error_Obs': daily_error_obs,
        'Daily_Error_Resid': daily_error_resid,
        'Daily_Error_Ens': daily_error_ens,
        'Daily_Error_Random': daily_sigma_rand,
        'Daily_Error_Ustar': daily_sigma_ustar,
        'Daily_Error_Total': daily_sigma_total,
        'Daily_CI_95_HalfWidth': daily_ci_halfwidth,
        'Days': n_days
    }


## N BUDGET

In [ ]:
# Settings
from plot_config import PERIODS
period_order = ['wheat_growing_season', 'wheat_treatment_period', 'post_wheat_harvest_period']
periods = {k: PERIODS[k] for k in period_order}
views = ['footprint', 'parcelA', 'parcelB']
qc_versions = ['QCF']

results = []

for period_name, (start, end) in periods.items():
    print(f"=== {period_name.upper()}: {start} to {end} ===")
    for v in views:
        print(f"--- {v} ---")
        for q in qc_versions:
            print(f"Processing QC Version: {q}")
            res = calculate_seasonal_budget(
                data,
                start, end,
                target_base='FN2O',
                model_type='XGBoost',
                view=v,
                qc=q,
                factor=60 * 30 * 28.014 * 1e-9 * 1e-3 * 1e4  # nmol N2O m-2 s-1 → kg N2O-N ha-1 per 30 min
            )
            res['Period'] = period_name 
            results.append(res)
    print("") 

# Create DataFrame
budget_N = pd.DataFrame(results)
budget_N['Period'] = pd.Categorical(budget_N['Period'], categories=period_order, ordered=True)

# Display nicely
print('-'*100)
print("\nFINAL BUDGET RESULTS (seasonal + daily mean):")
cols_show = [
    'Period', 'View', 'QC', 'Days',
    'Budget', 'CI_95_HalfWidth',
    'Error_Random', 'Error_Ustar', 'Error_Total',
    'Daily_Budget', 'Daily_CI_95_HalfWidth'
]

# Create the combined, formatted string column: budget +/- half-width
budget_N['Budget_Formatted'] = budget_N.apply(
    lambda row: f"{row['Budget']:.2f} \u00b1 {row['CI_95_HalfWidth']:.2f}", 
    axis=1
)
# Select just the context columns and your new formatted column
print_df = budget_N[['Period', 'View', 'Days', 'Budget_Formatted']].sort_values(by=['Period', 'View'])
# Print it as a clean text table without the row numbers (index)
print('-' * 60)
print(print_df.to_string(index=False))
print('-' * 60)


### UNCERTAINTY BREAKDOWN (N Budget)

In [ ]:
# Which uncertainty component dominates the N budget's total variance.
# Obs = measurement random error, Resid = gap-filling residual scale,
# Ens = ensemble/structural (correlation-aware, cumulative), Ustar = u* threshold choice.
# The four %'s sum to 100 for each row -- and Error_Ens is usually the big one,
# since a naive pointwise treatment would badly understate it (see Error_Ens
# vs. what a plain quadrature sum of the pointwise sigma would have given).
breakdown_N_df = budget_N[['Period', 'View']].copy()
for comp in ['Obs', 'Resid', 'Ens', 'Ustar']:
    breakdown_N_df[f'{comp}_%'] = (budget_N[f'VarFrac_{comp}'] * 100).round(1)

print('-' * 70)
print(breakdown_N_df.sort_values(by=['Period', 'View']).to_string(index=False))
print('-' * 70)

# Stacked bar: absolute error contribution (sqrt of variance share) per
# view, one panel per period, so the ensemble term's dominance is visible
# at a glance rather than buried in a table.
periods_list = list(budget_N['Period'].unique())
fig, axes = plt.subplots(1, len(periods_list), figsize=(4.5 * len(periods_list), 4), sharey=True)
if len(periods_list) == 1:
    axes = [axes]

components = ['Error_Obs', 'Error_Resid', 'Error_Ens', 'Error_Ustar']
colors = {'Error_Obs': '#56B4E9', 'Error_Resid': '#E69F00', 'Error_Ens': '#D55E00', 'Error_Ustar': '#009E73'}

for ax, period in zip(axes, periods_list):
    sub = budget_N[budget_N['Period'] == period].set_index('View').reindex(views)
    bottom = np.zeros(len(sub))
    for comp in components:
        ax.bar(sub.index, sub[comp], bottom=bottom, label=comp.replace('Error_', ''), color=colors[comp])
        bottom += sub[comp].to_numpy()
    ax.set_title(period, fontsize=10)
    ax.tick_params(axis='x', rotation=30)

axes[0].set_ylabel('Uncertainty contribution (kg N2O-N ha$^{-1}$)')
axes[-1].legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()


## EMISSION FACTORS

In [ ]:
# Define in-season N inputs for each treatment parcel (sum of all fertilization events during the treatment period) 
n_inputs = {
    'parcelA': 30 + 55.9 + 65.8,
    'parcelB': 32 + 79.7 + 33.4
    }

# Define the columns to calculate: budget value and its 95% CI half-width
cols_to_calc = {
    'Budget': 'EF_Value',
    'CI_95_HalfWidth': 'EF_HalfWidth'
}

# Define the period I want to calculate EF for
target_period = 'wheat_treatment_period'

rows = []
for parcel, n_in in n_inputs.items():
    parcel_data = budget_N.loc[
        (budget_N['View'] == parcel) &
        (budget_N['Period'] == target_period)
    ]
    if parcel_data.empty:
        raise ValueError(f"No budget_N rows found for View={parcel} and Period={target_period}")
    if len(parcel_data) > 1:
        raise ValueError(f"Multiple budget_N rows found for View={parcel} and Period={target_period}; expected 1")
    row = {'Period': target_period, 'Parcel': parcel, 'N_Input': n_in}
    for col_budget, col_res in cols_to_calc.items():
        budget_val = parcel_data[col_budget].iloc[0]
        row[col_res] = budget_val / n_in * 100
    rows.append(row)

ef_df = pd.DataFrame(rows).sort_values(['Period', 'Parcel']).reset_index(drop=True)

ef_df['EF_Formatted'] = ef_df.apply(
    lambda r: f"{r['EF_Value']:.2f} \u00b1 {r['EF_HalfWidth']:.2f}",
    axis=1
)

print_df = ef_df[['Period', 'Parcel', 'N_Input', 'EF_Formatted']]
print('-' * 60)
print(print_df.to_string(index=False))
print('-' * 60)


## GHG BUDGET

In [ ]:
T = 60 * 30  # seconds per 30 min

MW_CH4 = 16.04    # g/mol
MW_N2O = 44.013   # g/mol
MW_CO2 = 44.001    # g/mol

GWP100_CH4 = 27
GWP100_N2O = 273

factor_CH4_to_CO2_eq = T * MW_CH4 * 1e-9 * GWP100_CH4   # nmol -> mol
factor_N2O_to_CO2_eq = T * MW_N2O * 1e-9 * GWP100_N2O   # nmol -> mol
factor_NEE_to_CO2_eq = T * MW_CO2 * 1e-6   # µmol -> mol

gas_configs = [
    {'gas': 'N2O', 'target': 'FN2O', 'factor': factor_N2O_to_CO2_eq},
    {'gas': 'CH4', 'target': 'FCH4', 'factor': factor_CH4_to_CO2_eq},
    {'gas': 'CO2', 'target': 'NEE',  'factor': factor_NEE_to_CO2_eq}
]

qc_versions = ['QCF']

### EACH GAS

In [ ]:
results = []

for period_name, (start, end) in periods.items():
    print(f"=== Calculating budgets for {period_name.upper()}: {start} to {end} ===")
    for v in views:
        for q in qc_versions:
            # Inner Loop: Gases (runs calculate_seasonal_budget for N2O, CH4, and CO2) in CO2-equivalent units
            for config in gas_configs:
                res = calculate_seasonal_budget(
                    data, start, end,
                    target_base=config['target'],
                    model_type='XGBoost', 
                    view=v, 
                    qc=q,
                    factor=config['factor']
                )
                res['Period'] = period_name 
                res['Gas'] = config['gas']
                results.append(res)
print()

# Create final DataFrame
budget_df = pd.DataFrame(results)
budget_df['Period'] = pd.Categorical(budget_df['Period'], categories=period_order, ordered=True)

# Rearrange columns so Period, Gas, View, and QC are at the front
front_cols = ['Period', 'Gas', 'View', 'QC']
other_cols = [c for c in budget_df.columns if c not in front_cols]
budget_df = budget_df[front_cols + other_cols]


# Create the combined, formatted string column: budget +/- half-width
budget_df['Budget_Formatted'] = budget_df.apply(
    lambda row: f"{row['Budget']:.1f} \u00b1 {row['CI_95_HalfWidth']:.1f}", 
    axis=1
)
# Select just the context columns and your new formatted column
print_df = budget_df[['Gas', 'Period', 'View', 'Days', 'Budget_Formatted']].sort_values(by=['Gas', 'Period', 'View'])
# Print it as a clean text table without the row numbers (index)
print('-' * 60)
print(print_df.to_string(index=False))
print('-' * 60)


### UNCERTAINTY BREAKDOWN (per gas)

In [ ]:
# Which uncertainty component dominates, per gas / view / period.
# Obs = measurement random error, Resid = gap-filling residual scale,
# Ens = ensemble/structural (correlation-aware, cumulative), Ustar = u* threshold choice.
# The four %'s sum to 100 for each row.
breakdown_gas_df = budget_df[['Gas', 'Period', 'View']].copy()
for comp in ['Obs', 'Resid', 'Ens', 'Ustar']:
    breakdown_gas_df[f'{comp}_%'] = (budget_df[f'VarFrac_{comp}'] * 100).round(1)

print('-' * 80)
print(breakdown_gas_df.sort_values(by=['Gas', 'Period', 'View']).to_string(index=False))
print('-' * 80)


### TOTAL GHG

In [ ]:
total_ghg = (budget_df
           .groupby(['View','QC', 'Period'], as_index=False)
           .agg(
               Days=('Days','first'),
               GHG_CO2eq=('Budget','sum'),
               # quadrature across gases -- each component separately, so the
               # decomposition survives the aggregation (all four component
               # sums, combined in quadrature, still equal GHG_CO2eq_sigma)
               GHG_CO2eq_Error_Obs=('Error_Obs', lambda s: np.sqrt(np.sum(s**2))),
               GHG_CO2eq_Error_Resid=('Error_Resid', lambda s: np.sqrt(np.sum(s**2))),
               GHG_CO2eq_Error_Ens=('Error_Ens', lambda s: np.sqrt(np.sum(s**2))),
               GHG_CO2eq_Error_Ustar=('Error_Ustar', lambda s: np.sqrt(np.sum(s**2))),
               GHG_CO2eq_sigma=('Error_Total', lambda s: np.sqrt(np.sum(s**2)))
           ))

total_ghg['GHG_CO2eq_HalfWidth'] = 1.96 * total_ghg['GHG_CO2eq_sigma']
total_ghg['GHG_CO2eq_daily'] = total_ghg['GHG_CO2eq'] / total_ghg['Days']
total_ghg['GHG_CO2eq_daily_HalfWidth'] = total_ghg['GHG_CO2eq_HalfWidth'] / total_ghg['Days']

# Variance-fraction breakdown at the TOTAL-GHG level (sums to 1.0)
total_var = total_ghg['GHG_CO2eq_sigma'] ** 2
total_ghg['VarFrac_Obs']   = total_ghg['GHG_CO2eq_Error_Obs']**2   / total_var
total_ghg['VarFrac_Resid'] = total_ghg['GHG_CO2eq_Error_Resid']**2 / total_var
total_ghg['VarFrac_Ens']   = total_ghg['GHG_CO2eq_Error_Ens']**2   / total_var
total_ghg['VarFrac_Ustar'] = total_ghg['GHG_CO2eq_Error_Ustar']**2 / total_var

total_ghg = total_ghg.sort_values(by=['Period', 'View']).reset_index(drop=True)

# Create the combined, formatted string column: budget +/- half-width
total_ghg['GHG_Formatted'] = total_ghg.apply(
    lambda row: f"{row['GHG_CO2eq']:.0f} \u00b1 {row['GHG_CO2eq_HalfWidth']:.0f}", 
    axis=1
)

# Select just the context columns and your new formatted column
print_df = total_ghg[['Period', 'View', 'Days', 'GHG_Formatted']]

# Print it as a clean text table without the row numbers (index)
print('-' * 60)
print(print_df.to_string(index=False))
print('-' * 60)

# Uncertainty breakdown for the combined GHG total: which component of
# uncertainty (measurement, gap-filling residual, ensemble/structural, or
# u* threshold) actually dominates once you sum across gases
print("\nUNCERTAINTY BREAKDOWN (% of total variance, combined across N2O+CH4+CO2):")
breakdown_df = total_ghg[['Period', 'View']].copy()
for comp in ['Obs', 'Resid', 'Ens', 'Ustar']:
    breakdown_df[f'{comp}_%'] = (total_ghg[f'VarFrac_{comp}'] * 100).round(1)
print('-' * 70)
print(breakdown_df.sort_values(by=['Period', 'View']).to_string(index=False))
print('-' * 70)
